# The colour temperature of MoM-BH\*-1

**Goal:** a second estimate of the temperature of MoM-BH\*-1's glowing surface that does not use the spectrum at all. It uses only two published brightness measurements: one at 4.4 µm (NIRCam filter F444W) and one at 7.7 µm (MIRI filter F770W).

**The idea:** the *colour* of a blackbody (how bright it is at one wavelength compared with another) depends only on its temperature. A hotter object is relatively brighter at the shorter wavelength. So the ratio of the two brightnesses gives a temperature. Astronomers call this a **colour temperature**.

**What to compare with:** your blackbody fit to the spectrum gave about **4,970 K**. If the two methods agree within their error bars, the temperature is more believable.

**The two measurements** (Naidu et al. 2026a, the discovery paper, Extended Data Table 2):
- F444W: **243 ± 12 nJy** (NIRCam images, August 2023)
- F770W: **465 ± 52 nJy** (MIRI images, January 2023)

**Before you cite these, look both numbers up yourself** in the paper's photometry table (the paper is free to read: arXiv:2503.16596).

**The steps:**
- Step 1: the two measurements.
- Step 2: from colour to temperature.
- Step 3: the error bar.
- Step 4: how much do the choices matter?
- Step 5: see it.

**How to use:** run the grey code cells from top to bottom with **Shift + Enter** (or *Runtime → Run all*). Nothing is downloaded, and it takes a few seconds.

## Step 1 — The two measurements

This cell stores the two brightnesses, the wavelength that stands for each filter, and the plot colours. Then it prints the colour: the brightness at 7.7 µm divided by the brightness at 4.4 µm.

**Wavelengths:** MoM-BH\*-1 is at redshift z = 7.76, so its light has been stretched by a factor 1 + z = 8.76 on the way to us. Light we receive at 4.40 µm left the object at 4.40 / 8.76 = 0.502 µm (5,023 Å, green light). Light we receive at 7.64 µm left it at 8,721 Å (just beyond red, in the near-infrared). Both are on the red side of the Balmer break, where the glowing surface dominates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.optimize import brentq
from scipy.integrate import quad

Z = 7.76                              # redshift of MoM-BH*-1
HC_OVER_K = 1.4388e8                  # Planck's constant x speed of light / Boltzmann's constant, in Å·K

# --- the two published measurements (Naidu et al. 2026a, Extended Data Table 2), in nJy ---
F444W, F444W_ERR = 243.0, 12.0        # NIRCam, August 2023
F770W, F770W_ERR = 465.0, 52.0        # MIRI, January 2023

# --- the wavelength that stands for each filter (its "pivot" wavelength), in micrometres ---
W444, W770 = 4.40, 7.64

T_SPECTRUM_FIT = 4970                 # K: your blackbody fit to the spectrum (temperature notebook)

# --- plot style and colours (same as the stage notebooks) ---
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#0b0b0b", "axes.titlecolor": "#0b0b0b", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "font.size": 11, "legend.frameon": False, "legend.fontsize": 9.5})
BLUE = "#2a78d6"
INK, INK2, MUTED, ZONE, SURFACE = "#0b0b0b", "#52514e", "#898781", "#ebeae5", "#fcfcfb"

ratio = F770W / F444W
ratio_err = ratio * np.hypot(F770W_ERR / F770W, F444W_ERR / F444W)   # the two error bars combined

print(f"Wavelengths in the object's own frame: {W444*1e4/(1+Z):,.0f} Å and {W770*1e4/(1+Z):,.0f} Å")
print(f"Colour (F770W ÷ F444W): {ratio:.2f} ± {ratio_err:.2f}")
print("So MoM-BH*-1 is about twice as bright at 7.7 µm as at 4.4 µm.")

## Step 2 — From colour to temperature

**Planck's law** gives the brightness of a blackbody at every wavelength λ for a temperature T:

brightness ∝ (1 / λ³) × 1 / (e^(hc / λkT) − 1)

Divide the brightness at one wavelength by the brightness at another, and everything except the temperature cancels out: the size of the object, its distance, all of it. That is why a colour gives a temperature.

The cell does two things:
1. `predicted_ratio(T)` works out the colour a blackbody of temperature T would have.
2. `colour_temperature(ratio)` runs that backwards: it searches for the temperature whose predicted colour equals the measured one.

In [ ]:
def blackbody(lam_um, T):
    """Blackbody brightness (per unit frequency) at OBSERVED wavelength lam_um, in micrometres,
    for a surface of temperature T (K) at redshift Z. Only ratios of this are ever used."""
    lam_rest = lam_um * 1e4 / (1 + Z)                 # Å, in the object's own frame
    return lam_rest**-3 / np.expm1(HC_OVER_K / (lam_rest * T))


def predicted_ratio(T, w_long=W770, w_short=W444):
    """The colour (long wavelength ÷ short wavelength) of a blackbody of temperature T."""
    return blackbody(w_long, T) / blackbody(w_short, T)


def colour_temperature(measured_ratio, w_long=W770, w_short=W444):
    """The temperature (K) whose predicted colour equals the measured one."""
    return brentq(lambda T: predicted_ratio(T, w_long, w_short) - measured_ratio, 1500, 30000)


T_colour = colour_temperature(ratio)

print("What colour would a blackbody have?")
for T in [4000, 5000, 6000, 7000]:
    print(f"   {T:,} K  ->  F770W ÷ F444W = {predicted_ratio(T):.2f}")
print(f"\nMeasured colour: {ratio:.2f}")
print(f"Colour temperature: {T_colour:,.0f} K")

## Step 3 — The error bar

Both brightnesses have error bars, so the colour does too, and so does the temperature.

**How the cell does it:** it draws 200,000 possible pairs of brightnesses inside their error bars, turns each pair into a temperature, and looks at the spread. The **middle value** and the **range holding the middle 68%** of the results are your answer. This is the same method as in stage 5.

**Which way does it go?** More light at 7.7 µm means a redder, cooler object. So the *top* of the F770W error bar gives the *lowest* temperature.

In [ ]:
rng = np.random.default_rng(7)
N_DRAW = 200_000

ratio_draws = rng.normal(F770W, F770W_ERR, N_DRAW) / rng.normal(F444W, F444W_ERR, N_DRAW)

# turn every drawn colour into a temperature (a look-up table is much faster than 200,000 searches)
T_grid = np.linspace(2500, 15000, 5001)
T_draws = np.interp(ratio_draws, predicted_ratio(T_grid)[::-1], T_grid[::-1])

T_low, T_mid, T_high = np.percentile(T_draws, [16, 50, 84])

print(f"Colour temperature: {T_mid:,.0f} K   (range {T_low:,.0f} - {T_high:,.0f} K)")
print(f"Rounded for your note: about {round(T_mid, -2):,.0f} K ({round(T_low, -2):,.0f} - {round(T_high, -2):,.0f} K)")

gap = T_mid - T_SPECTRUM_FIT
print(f"\nYour spectrum fit: {T_SPECTRUM_FIT:,} K. That is {gap:,.0f} K lower, "
      f"or {gap / (T_mid - T_low):.1f} times the lower error bar.")
print(f"Colour a {T_SPECTRUM_FIT:,} K blackbody would have: {predicted_ratio(T_SPECTRUM_FIT):.2f} "
      f"(measured: {ratio:.2f} ± {ratio_err:.2f})")

## Step 4 — How much do the choices matter?

A measurement always involves choices. This cell repeats Step 2 with each choice changed, one at a time:

1. **Which wavelength stands for F444W?** The filter is wide (3.9–5.0 µm). Step 2 used 4.40 µm; some tables give 4.44 µm.
2. **One wavelength, or the whole filter?** Instead of one wavelength per filter, average the blackbody over the full width of each filter (treated as a simple box).
3. **The rounded magnitude.** The paper's text gives F444W as "25.4 mag", which is 251 nJy. Your earlier notebooks used the temperature that follows from this rounded value (about 5,500 K). The table value, 243 nJy, is the more exact one.
4. **Emission lines.** The Hβ and [O III] lines fall inside the F444W filter. They add light that does not come from the glowing surface. If some per cent of the F444W light is line emission, the surface is fainter there than the filter says.
5. **The two images were taken seven months apart** (about 24 days in the object's own time, because of the redshift). The paper reports a hint that the object varies: a 30 (+7/−7)% brightening in 56 days at 3–5 µm, "albeit measured with different instruments". If its brightness changed between January and August 2023, the colour is off.

Choices 1–3 are yours. Choices 4 and 5 are things about the object that this notebook cannot measure, so the cell shows "what if" cases.

In [ ]:
def band_average(T, lo_um, hi_um):
    """Average blackbody brightness over a filter running from lo_um to hi_um (a simple box shape)."""
    total = quad(lambda w: blackbody(w, T) / w, lo_um, hi_um)[0]
    return total / np.log(hi_um / lo_um)


def colour_temperature_whole_filter(measured_ratio):
    # approximate filter edges in micrometres: F770W 6.58-8.69, F444W 3.88-4.98
    f = lambda T: band_average(T, 6.58, 8.69) / band_average(T, 3.88, 4.98) - measured_ratio
    return brentq(f, 1500, 30000)


def show(label, T):
    print(f"   {label:<62} {T:>6,.0f} K   ({T - T_colour:+5.0f} K)")


print(f"Step 2 result: {T_colour:,.0f} K.  With one choice changed at a time:\n")

print("Your choices")
show("F444W stands at 4.44 µm instead of 4.40 µm", colour_temperature(ratio, w_short=4.44))
show("averaged over the whole width of each filter", colour_temperature_whole_filter(ratio))
show("rounded magnitude: F444W = 25.4 mag = 251 nJy", colour_temperature(F770W / 251.0))

print("\nWhat if part of the F444W light is emission lines?")
for percent in [2, 5, 10]:
    show(f"{percent}% of the F444W light is line emission", colour_temperature(F770W / (F444W * (1 - percent / 100))))

print("\nWhat if the object changed between January and August 2023?")
for percent in [10, 30]:
    show(f"it was {percent}% brighter in August than in January", colour_temperature(F770W / (F444W / (1 + percent / 100))))
    show(f"it was {percent}% fainter in August than in January", colour_temperature(F770W / (F444W / (1 - percent / 100))))

## Step 5 — See it

The plot shows the whole method in one picture:
- **Blue curve:** the colour a blackbody would have at each temperature (Step 2).
- **Grey band:** the measured colour and its error bar (Step 1).
- **Where they cross** is the colour temperature and its range (Step 3), marked on the bottom axis.
- **Open circle:** your spectrum fit. It sits on the curve at 4,970 K, a little above the grey band.

The plot is saved as **MoM-BH1_colour_temperature.png** (file panel on the left in Colab).

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.7))

T_plot = np.linspace(3800, 7600, 400)
ax.plot(T_plot, predicted_ratio(T_plot), color=BLUE)

# the measurement: a band for the error bar, a line for the value
ax.axhspan(ratio - ratio_err, ratio + ratio_err, color=ZONE, zorder=0)
ax.axhline(ratio, color=INK2, lw=1)

# the answer, dropped down to the temperature axis
y_floor = 0.9
for T, style in [(T_low, ":"), (T_mid, "-"), (T_high, ":")]:
    ax.plot([T, T], [y_floor, predicted_ratio(T)], color=INK2, lw=1, ls=style)
ax.plot(T_mid, ratio, "o", ms=8, color=BLUE, mec=SURFACE, mew=2, zorder=5)

# your spectrum fit, for comparison
ax.plot(T_SPECTRUM_FIT, predicted_ratio(T_SPECTRUM_FIT), "o", ms=8, mfc=SURFACE, mec=INK, mew=1.6, zorder=5)

# direct labels (no legend needed: one curve); the small background keeps grid lines out of the text
box = dict(facecolor=SURFACE, edgecolor="none", pad=1.5)
ax.text(7550, ratio + ratio_err + 0.04, f"measured colour: {ratio:.2f} ± {ratio_err:.2f}",
        ha="right", va="bottom", color=INK2, fontsize=10)
ax.text(4530, predicted_ratio(4450), "colour of a blackbody", ha="left", va="center", color=INK2, fontsize=10, bbox=box)
ax.text(T_SPECTRUM_FIT + 90, predicted_ratio(T_SPECTRUM_FIT) + 0.06, f"your spectrum fit\n{T_SPECTRUM_FIT:,} K",
        ha="left", va="bottom", color=INK2, fontsize=10, bbox=box)
ax.text(T_high + 70, y_floor + 0.05, f"{T_mid:,.0f} K  (range {T_low:,.0f} - {T_high:,.0f} K)",
        ha="left", va="bottom", color=INK, fontsize=10.5, bbox=box)

ax.set_xlim(3800, 7600)
ax.set_ylim(y_floor, 3.6)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_xlabel("Temperature (K)")
ax.set_ylabel("Brightness at 7.7 µm ÷ brightness at 4.4 µm")
ax.set_title("The colour of MoM-BH*-1 gives its temperature", loc="left", fontsize=12.5)

plt.tight_layout()
plt.savefig("MoM-BH1_colour_temperature.png", dpi=200)
plt.show()

# What the colour tells you

- **The colour temperature is about 5,400 K (5,100–5,700 K).** Your spectrum fit, 4,970 K, is lower by about one and a half error bars. That is fair agreement for two methods this different. Both are well below the 6,465 K that Curtis et al. (2026) find when they allow for dust.
- **Your adopted value stays.** 5,200 ± 300 K covers both of your estimates, as before.
- **Your earlier notebooks say 5,500 K.** That came from the rounded magnitude (Step 4, choice 3). The exact table value gives about 5,400 K. The difference is much smaller than the error bar, so nothing that follows from 5,200 ± 300 K changes. But the sentence in your note should quote the number from this notebook.

**Number for your note** (copy it from your own Step 3 output): the colour temperature and its range, rounded to the nearest 100 K.

**How far to trust it:**
- **The object may have changed between the two images** (Step 4, last block). A 10% change moves the answer by about 250 K. This is the main reason the error bar on your adopted temperature is not smaller.
- **Emission lines inside F444W push the answer down** by roughly 50 K for every 2% of line light (Step 4). You could measure that percentage from the spectrum.
- **Dust would make the object look cooler than it is.** This estimate assumes no dust. Dust dims short wavelengths more than long ones, so a hotter surface behind dust can show the same colour. Curtis et al. (2026) allow for dust, which is one reason they find a hotter surface.
- **A blackbody is only an approximation.** A real atmosphere has absorption that changes the colour.